# Fusion-bind Tier-1 tests

Five cases from `docs/superpowers/specs/2026-04-17-own-the-stack-design.md` Tier 1.
Run order matters - the helpers cell defines `fake_impl`,
`nvfp4_o_proj`, etc. used by all tests.

In [ ]:
import sys

sys.path.insert(0, "/home/natfii/docker/nvllm")
from unittest.mock import MagicMock, patch

import torch

from vllm.v1.attention.backends.cute_paged._backend import (
    CutePagedAttentionImpl,
)


def fake_impl():
    impl = CutePagedAttentionImpl.__new__(CutePagedAttentionImpl)
    impl.num_heads = 24
    impl.head_size = 256
    impl.num_kv_heads = 4
    impl.scale = 1.0 / (256**0.5)
    impl.kv_cache_dtype = "auto"
    impl.num_queries_per_kv = 6
    impl.kv_sharing_target_layer_name = None
    impl.alibi_slopes = None
    impl.sliding_window = None
    impl.logits_soft_cap = None
    impl._fusion_bound = False
    impl._fusion_active = False
    impl._fusion_attached = False
    return impl


def nvfp4_o_proj(hidden_dim=5120, q_size=6144, device="cpu"):
    mod = MagicMock()
    # NVFP4 packed weight is uint8 — not differentiable, so use plain tensor.
    mod.weight = torch.zeros(hidden_dim, q_size // 2, dtype=torch.uint8, device=device)
    # FP8 block scales likewise cannot be nn.Parameter (no grad support for fp8).
    mod.weight_scale = torch.zeros(
        hidden_dim, q_size // 16, dtype=torch.float8_e4m3fn, device=device
    )
    # Global scale IS fp32 and IS a Parameter in the real quant module.
    mod.weight_global_scale = torch.nn.Parameter(
        torch.tensor([1.0], dtype=torch.float32, device=device)
    )
    return mod


def bf16_o_proj(hidden_dim=5120, q_size=6144, device="cpu"):
    mod = MagicMock(spec=["weight"])
    mod.weight = torch.nn.Parameter(
        torch.zeros(hidden_dim, q_size, dtype=torch.bfloat16, device=device)
    )
    return mod


def post_norm(hidden_dim=5120, device="cpu"):
    mod = MagicMock()
    mod.weight = torch.nn.Parameter(
        torch.ones(hidden_dim, dtype=torch.bfloat16, device=device)
    )
    mod.variance_epsilon = 1e-6
    return mod


def parent_layer(
    o_proj,
    post_norm_mod,
    prefix="model.layers.0",
    attn_output_gate=True,
    hidden_size=5120,
    num_heads=24,
    head_dim=256,
):
    layer = MagicMock()
    layer.prefix = prefix
    layer.self_attn = MagicMock()
    layer.self_attn.o_proj = o_proj
    layer.self_attn.num_heads = num_heads
    layer.self_attn.head_dim = head_dim
    layer.self_attn.hidden_size = hidden_size
    layer.self_attn.attn_output_gate = attn_output_gate
    layer.post_attention_layernorm = post_norm_mod
    return layer


def mock_cfg(max_num_seqs=16):
    cfg = MagicMock()
    cfg.scheduler_config.max_num_seqs = max_num_seqs
    return cfg


# Buffer allocation in attach_fusion uses 'cuda' device by default. Patch to
# 'cpu' so these tests run on host without a GPU context.
_orig_prealloc = CutePagedAttentionImpl._preallocate_fusion_buffers


def _cpu_prealloc(self, max_num_seqs, hidden_dim, q_size, device):
    return _orig_prealloc(self, max_num_seqs, hidden_dim, q_size, "cpu")


print("test helpers ready")

## Test 1 — NVFP4 happy-path

In [ ]:
impl = fake_impl()
o = nvfp4_o_proj()
n = post_norm()
p = parent_layer(o, n)

with (
    patch("vllm.config.get_current_vllm_config", return_value=mock_cfg(16)),
    patch.object(CutePagedAttentionImpl, "_preallocate_fusion_buffers", _cpu_prealloc),
):
    impl.attach_fusion(p)

assert impl._fusion_attached is True
assert impl._fusion_max_num_seqs == 16
assert impl._fusion_hidden_dim == 5120
assert impl._fusion_q_size == 24 * 256

impl._resolve_fusion_weights()
assert impl._fusion_bound is True
assert impl.wo_weight is o.weight
assert impl.wo_global_scale is o.weight_global_scale
assert impl.rmsnorm_gamma is n.weight
assert impl.rmsnorm_eps == 1e-6
print("Test 1 PASS")

## Test 2 — BF16 skip-path

In [ ]:
impl = fake_impl()
o = bf16_o_proj()
n = post_norm()
p = parent_layer(o, n)

with (
    patch("vllm.config.get_current_vllm_config", return_value=mock_cfg(16)),
    patch.object(CutePagedAttentionImpl, "_preallocate_fusion_buffers", _cpu_prealloc),
):
    impl.attach_fusion(p)

impl._resolve_fusion_weights()  # must not raise AttributeError
assert impl._fusion_bound is False
assert impl._fusion_attached is True
print("Test 2 PASS")

## Test 3 — Double-resolve rebinds to fresh tensor identity

In [ ]:
impl = fake_impl()
o = nvfp4_o_proj()
n = post_norm()
p = parent_layer(o, n)

with (
    patch("vllm.config.get_current_vllm_config", return_value=mock_cfg(16)),
    patch.object(CutePagedAttentionImpl, "_preallocate_fusion_buffers", _cpu_prealloc),
):
    impl.attach_fusion(p)

impl._resolve_fusion_weights()
old_gs = impl.wo_global_scale

o.weight_global_scale = torch.nn.Parameter(torch.tensor([2.0], dtype=torch.float32))

impl._resolve_fusion_weights()
assert impl.wo_global_scale is o.weight_global_scale
assert impl.wo_global_scale is not old_gs
assert impl.wo_global_scale.item() == 2.0
print("Test 3 PASS")

## Test 4 — Buffer pointer stability across attach calls

In [ ]:
impl = fake_impl()
o = nvfp4_o_proj()
n = post_norm()
p = parent_layer(o, n)

with (
    patch("vllm.config.get_current_vllm_config", return_value=mock_cfg(16)),
    patch.object(CutePagedAttentionImpl, "_preallocate_fusion_buffers", _cpu_prealloc),
):
    impl.attach_fusion(p)

ptrs_before = (
    impl.wo_output.data_ptr(),
    impl.rmsnorm_output.data_ptr(),
    impl.gate_buf.data_ptr(),
)

with (
    patch("vllm.config.get_current_vllm_config", return_value=mock_cfg(16)),
    patch.object(CutePagedAttentionImpl, "_preallocate_fusion_buffers", _cpu_prealloc),
):
    impl.attach_fusion(p)

ptrs_after = (
    impl.wo_output.data_ptr(),
    impl.rmsnorm_output.data_ptr(),
    impl.gate_buf.data_ptr(),
)

assert ptrs_before == ptrs_after, f"{ptrs_before} != {ptrs_after}"
print("Test 4 PASS")

## Test 5 — Per-forward gate boundary

In [ ]:
impl = fake_impl()
o = nvfp4_o_proj()
n = post_norm()
p = parent_layer(o, n)

with (
    patch("vllm.config.get_current_vllm_config", return_value=mock_cfg(16)),
    patch.object(CutePagedAttentionImpl, "_preallocate_fusion_buffers", _cpu_prealloc),
):
    impl.attach_fusion(p)

impl._resolve_fusion_weights()
assert impl._fusion_bound is True


def gate_decision(nat, is_decode_only):
    fits = nat <= getattr(impl, "_fusion_max_num_seqs", 0)
    return impl._fusion_bound and is_decode_only and fits


assert gate_decision(8, True) is True, "decode within cap -> fuse"
assert gate_decision(17, True) is False, "decode over cap -> no fuse (A3)"
assert gate_decision(8, False) is False, "prefill -> no fuse"
print("Test 5 PASS")